In [ ]:
# Control-composition sensitivity review (Moore et al. Batch_1)
# Reads only what run_control_composition.py already wrote to Analysis_Results/Control_Composition/
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, '.')
from viz_style import apply_style
apply_style()
import config

res = pd.read_csv(config.CTRL_COMP_DIR / 'jaccard_results.csv')
null = pd.read_csv(config.CTRL_COMP_DIR / 'null_distribution.csv')
print(f"tertile rows: {len(res)}, null rows: {len(null)}")
res.head()

In [ ]:
# 1. null p-value per (disease, layer, axis): how often a size-matched random split
# agrees no better than the bias-stratified split
def null_pvalue(res, null, metric='jaccard_k100'):
    out = []
    for (dis, lay), g in null.groupby(['disease', 'layer']):
        ref = g[metric].values
        for _, r in res[(res.disease == dis) & (res.layer == lay)].iterrows():
            out.append(dict(disease=dis, layer=lay, axis=r['axis'], obs=r[metric],
                            null_mean=float(ref.mean()), null_sd=float(ref.std()),
                            p_lower=float((ref <= r[metric]).mean())))
    return pd.DataFrame(out)

pval = null_pvalue(res, null)
pval.sort_values('p_lower').head(20)

In [ ]:
# 2. layer x axis heatmap of jaccard_k100 (tertile) vs null mean
for disease, g in res.groupby('disease'):
    piv = g.pivot(index='axis', columns='layer', values='jaccard_k100')
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(piv.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=90)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    ax.set_title(f'{disease}: jaccard@100 (tertile split)')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    (config.CTRL_COMP_FIG_DIR).mkdir(parents=True, exist_ok=True)
    plt.savefig(config.CTRL_COMP_FIG_DIR / f'heatmap_jaccard100_{disease.replace(" ", "_")}.png', dpi=150)
    plt.show()

In [ ]:
# 3. dose-response: technical imbalance (delta cohen's d across strata) vs jaccard
layer_focus = ['CPM_log1p', 'TMM_log2', 'RUVg_Platelet_k2', 'Proposed_Full_k2']
fig, axes = plt.subplots(1, len(layer_focus), figsize=(4 * len(layer_focus), 4), sharey=True)
for ax, layer in zip(axes, layer_focus):
    sub = res[res.layer == layer]
    null_band = null[null.layer == layer]['jaccard_k100']
    ax.axhspan(null_band.quantile(0.05), null_band.quantile(0.95), color='#cccccc', alpha=0.5, label='null 90% band')
    for disease, marker in zip(sub['disease'].unique(), ['o', 's']):
        d = sub[sub.disease == disease]
        ax.scatter(d['delta_d'], d['jaccard_k100'], label=disease, marker=marker, s=25)
    ax.set_title(layer); ax.set_xlabel('max-min |cohen d| across strata')
axes[0].set_ylabel('jaccard@100'); axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(config.CTRL_COMP_FIG_DIR / 'dose_response.png', dpi=150)
plt.show()

In [ ]:
# 4. jaccard@K curve, tertile vs null, per layer (K sensitivity)
k_grid = [25, 50, 100, 200, 500]
fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharey=True)
for row, disease in enumerate(res['disease'].unique()):
    for col, layer in enumerate(['CPM_log1p', 'TMM_log2', 'RUVg_Platelet_k1', 'RUVg_Platelet_k3', 'Proposed_Full_k2']):
        ax = axes[row, col]
        r = res[(res.disease == disease) & (res.layer == layer)]
        n = null[(null.disease == disease) & (null.layer == layer)]
        ax.plot(k_grid, [r[f'jaccard_k{k}'].mean() for k in k_grid], 'o-', color='#d64545', label='tertile mean')
        ax.plot(k_grid, [n[f'jaccard_k{k}'].mean() for k in k_grid], 'o-', color='#888888', label='null mean')
        ax.fill_between(k_grid, [n[f'jaccard_k{k}'].quantile(0.05) for k in k_grid],
                        [n[f'jaccard_k{k}'].quantile(0.95) for k in k_grid], color='#888888', alpha=0.2)
        ax.set_title(f'{disease[:12]} / {layer}', fontsize=8)
        if row == 1: ax.set_xlabel('K')
axes[0, 0].set_ylabel('jaccard@K'); axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(config.CTRL_COMP_FIG_DIR / 'jaccard_k_curve.png', dpi=150)
plt.show()

In [ ]:
# 5. spearman rho across the whole ranked gene list, same layer x axis grid
for disease, g in res.groupby('disease'):
    piv = g.pivot(index='axis', columns='layer', values='spearman')
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(piv.values, aspect='auto', cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=90)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    ax.set_title(f'{disease}: spearman rho (full ranked list)')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(config.CTRL_COMP_FIG_DIR / f'heatmap_spearman_{disease.replace(" ", "_")}.png', dpi=150)
    plt.show()

In [ ]:
# 6. optional deep dive into one comparison's t-statistics / expression matrices
# (written per-comparison by run_control_composition.py; load only what's needed)
tag = 'Pancreatic_Cancer__PC1_bias'
stat_dir = config.CTRL_COMP_STAT_DIR / tag
t0 = pd.read_csv(stat_dir / 'T0_welch_t.csv.gz', index_col=0)
t2 = pd.read_csv(stat_dir / 'T2_welch_t.csv.gz', index_col=0)
t0[['CPM_log1p', 'GeneName']].join(t2[['CPM_log1p']], rsuffix='_T2').sort_values('CPM_log1p').head(10)